# 2. Document 与文档加载（精简）

来源：《尚硅谷-10-RAG.pdf》，第 7–24 页；对应课件 **2.1、2.2**。页码均指 PDF 阅读器中的页序。

整理方式：原理保留关键逻辑；第 2 章示例精简，重点学习拆分器源码与递归拆分。
只记录到 2.5.2，不收录 2.5.3 案例。

**运行约定**：在 `D:/grow/langchain/7_RAG` 打开本文件，选择
`D:/conda/envs/langchain/python.exe` 对应的 Python 内核；代码单元按本文件顺序运行。
各 Notebook 的变量相互独立。使用 `len` 的示例以 Python 字符长度计数，不能据此推断 Token 数。

## 2.1 环境与数据

课件完整依赖位于 `requirements_full.txt`；当前工程已有 `asset/load`，本笔记不重复安装整套依赖。
需要增加某个 Loader 时，再按它实际使用的包准备环境。

- 通用部分：`langchain-core`、`langchain-community`、`langchain-text-splitters`。
- PDF 文本：`pypdf`；JSON 的 `JSONLoader`：`jq`。
- Word、Markdown、HTML 的 Unstructured Loader：需要 `unstructured` 及对应格式依赖。

本文件的示例只读取本地 TXT 和文本型 PDF，不上传文件、不调用模型。

In [1]:
import sys
from pathlib import Path
from importlib.metadata import version

print("Python:", sys.executable)
print("langchain-community:", version("langchain-community"))
print("langchain-text-splitters:", version("langchain-text-splitters"))

# Notebook 位于 7_RAG；所有示例资料统一从这个目录定位。
ASSET = Path("D:/grow/langchain/7_RAG/asset/load")
print("资料目录存在:", ASSET.is_dir())

Python: d:\conda\envs\langchain\python.exe
langchain-community: 0.4.1
langchain-text-splitters: 1.1.1
资料目录存在: True


## 2.2 Document 是统一的数据载体

| 属性 | 类型 | 含义 |
| --- | --- | --- |
| `page_content` | `str` | 真正用于拆分、向量化和提供给模型的文本 |
| `metadata` | `dict` | 来源、页码、行号、标题等描述信息 |
| `id` | `str` 或 `None` | 可选文档标识，不等于所有向量库都会自动保留的数据库主键 |

Loader 统一返回 `Document`，所以不同文件格式能够进入相同的下游处理流程。
`Document` 用于检索数据；给聊天模型发送消息时仍使用 `HumanMessage` 等消息类型。

## 2.3 各种 Loader：知道用途和关键参数即可

| 数据格式 | 课件使用的组件 | 返回粒度 / 关键参数 |
| --- | --- | --- |
| TXT | `TextLoader` | 通常一文件一个 Document；指定正确的 `encoding`，GBK 与 UTF-8 不混用 |
| CSV | `CSVLoader` | 通常一行一个 Document；列内容组合成文本，元数据含 `source`、`row` |
| JSON | `JSONLoader` | 用 `jq_schema` 决定提取范围；若结果不是字符串，可设 `text_content=False` 转为文本 |
| PDF | `PyPDFLoader` | 默认按页加载；`extraction_mode="plain"` / `"layout"` 控制文本提取方式 |
| Word | `UnstructuredWordDocumentLoader` | `mode="single"` 聚合；`"elements"` 保留元素级结果 |
| Markdown | `UnstructuredMarkdownLoader` | 可按元素解析；加载与后续按标题拆分是两个步骤 |
| HTML | `UnstructuredHTMLLoader` | 提取页面元素；标题信息可供后续拆分使用 |
| 文件夹 | `DirectoryLoader` | `glob` 过滤文件；`loader_cls` 决定每个文件用什么 Loader，可配置多线程 |

JSON 的 `jq_schema` 记住几个形状：`.` 选整个对象，`.[]` 遍历数组，
`.messages[].content` 提取消息内容，`.data.items[]` 提取嵌套数组中的每一项。
一个 jq 输出项通常对应一个 Document；`text_content=False` 不会让 `page_content` 变成字典。

PDF 难点是扫描件、分栏、表格和公式。`PyPDFLoader` 的普通文本提取不等于 OCR。
课件的 MinerU 例子属于「上传文件 → 获取任务 → 获取解析结果」的外部解析流程，此处只记用途，省略长代码。

## 2.4 保留一个代表例子：TextLoader

重点观察 `list[Document]`、`page_content` 与 `metadata`。

In [2]:
from langchain_community.document_loaders import TextLoader

text_loader = TextLoader(
    file_path=str(ASSET / "01-langchain-utf-8.txt"),
    encoding="utf-8",
)
docs = text_loader.load()  # 一次性收集为 list[Document]

print("返回类型:", type(docs).__name__)
print("文档数量:", len(docs))
print("文档类型:", type(docs[0]).__name__)
print("正文:", docs[0].page_content)
print("元数据:", docs[0].metadata)

返回类型: list
文档数量: 1
文档类型: Document
正文: LangChain 是一个用于构建基于大语言模型（LLM）应用的开发框架，旨在帮助开发者更高效地集成、管理和增强大语言模型的能力，构建端到端的应用程序。它提供了一套模块化工具和接口，支持从简单的文本生成到复杂的多步骤推理任务
元数据: {'source': 'D:\\grow\\langchain\\7_RAG\\asset\\load\\01-langchain-utf-8.txt'}


## 2.5 load 与 lazy_load：精简源码关系

```python
# BaseLoader.load 的核心逻辑：将迭代器收集成列表。
def load(self):
    return list(self.lazy_load())
```

`lazy_load()` 返回迭代器，可以逐份消费文档，避免在调用方一次性持有所有返回文档。
实际能否逐页 / 逐元素解析、能节省多少内存，还取决于具体 Loader 的实现；
对仅返回一个 Document 的 TXT，不能把它理解成按文本行流式读取。
新写 Loader 通常实现 `lazy_load()`，再复用 `load()`。

课件同时介绍了 `load_and_split()`。学习时拆成 `loader.load()` 和
`splitter.split_documents(docs)`，更容易看清加载与拆分各自的职责。

In [3]:
# 与 load() 返回列表不同，lazy_load() 返回可迭代对象。
for doc in text_loader.lazy_load():
    print(type(doc).__name__, len(doc.page_content), doc.metadata)

Document 112 {'source': 'D:\\grow\\langchain\\7_RAG\\asset\\load\\01-langchain-utf-8.txt'}


## 2.6 PDF 只保留最短用法

`page` 元数据通常从 0 开始，显示给人看时可以再加 1。
下一份递归拆分笔记会复用同一份 PDF，观察页码如何保留到 Chunk。

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(str(ASSET / "04-load.pdf"), extraction_mode="plain")
pdf_docs = loader.load()
print("页数:", len(pdf_docs))
print("首页正文节选:", pdf_docs[0].page_content[:120])
print("页码:", pdf_docs[0].metadata["page"])
print("来源:", pdf_docs[0].metadata["source"])

页数: 1
首页正文节选: "他的车，他的命！ 他忽然想起来，一年，二年，至少有三四年；一滴汗，两滴汗，不
知道多少万滴汗，才挣出那辆车。从风里雨里的咬牙，从饭里茶里的自苦，才赚出那辆车。
那辆车是他的一切挣扎与困苦的总结果与报酬，像身经百战的武士的一颗徽章。……他老
页码: 0
来源: D:\grow\langchain\7_RAG\asset\load\04-load.pdf


**复习要点**：文件编码、一次加载与延迟加载、统一的 Document、按页 / 按行 / 按元素的不同粒度。
元数据要在加载和拆分时保留，后面才能定位检索结果的来源。

## 笔记导航

1. [7_1_RAG原理与流程](7_1_RAG原理与流程.ipynb)
2. [7_2_Document与文档加载](7_2_Document与文档加载.ipynb)
3. [7_3_TextSplitter源码与Character切分](7_3_TextSplitter源码与Character切分.ipynb)
4. [7_4_RecursiveCharacterTextSplitter详解](7_4_RecursiveCharacterTextSplitter详解.ipynb)
5. [7_5_其他拆分方式速记](7_5_其他拆分方式速记.ipynb)
6. [7_6_Embedding与向量存储](7_6_Embedding与向量存储.ipynb)